## <font color='red'> INSTRUCTIONS </font>

<b> 
1. Write your code only in cells below the "WRITE CODE BELOW" title. Do not modify the code below the "DO NOT MODIFY" title. <br>
2. The expected data types of the output answers for each question are given in the last cell through assertion statements. Your answers must match these expected output data types. Hint: Many of the answers need to be a Python dictionary. Consider methods like to_dict() to convert a Pandas Series to a dictionary. <br>
3. The answers are then written to a JSON file named my_results_PA1.json. You can compare this with the provided expected output file "expected_results_PA1.json". <br>
4. After you complete writing your code, click "Kernel -> Restart Kernel and Run All Cells" on the top toolbar. There should NOT be any syntax/runtime errors, otherwise points will be deducted. <br>
5. For submitting your solution, first download your notebook by clicking "File -> Download". Rename the file as &ltTEAM_ID&gt.ipynb" and upload to Canvas.</b>


## <font color='red'> DO NOT MODIFY </font>

In [1]:
import time
import json
import dask
import dask.dataframe as dd
import pandas as pd
import ast
import re
from dask.distributed import Client
import ctypes
import numpy as np

def trim_memory() -> int:
    """
    helps to fix any memory leaks.
    """
    libc = ctypes.CDLL("libc.so.6")
    return libc.malloc_trim(0)

client = Client("127.0.0.1:8786")
client.run(trim_memory)
client = client.restart()
print(client)

None


In [ ]:
start = time.time()

## <font color='blue'> WRITE CODE BELOW </font>

In [2]:
user_reviews = dd.read_csv('user_reviews.csv')
products = dd.read_csv('products.csv', dtype={'asin': 'object'})

num_reviews = len(user_reviews)
num_products = len(products)

In [ ]:
## Question 1
# Time: 30 seconds
# start = time.time()
null_percent = ((user_reviews.isna().sum() / num_reviews) * 100).round(2).compute()
# num_nulls = user_reviews.isna().sum().compute()
# num_nulls = np.round((num_nulls / user_reviews.shape[0]) * 100, 2)
ans1 = null_percent.to_dict()
# end = time.time()
ans1

In [ ]:
## Question 2
# Time: 18.55 seconds
# start = time.time()
null_percent = ((products.isna().sum() / num_products) * 100).round(2).compute()
ans2 = null_percent.to_dict()
# end = time.time()
ans2

In [24]:
# ## Question 3
# Time: 56.6 seconds
# start = time.time()
# Needed to speed this up, trying to index by asin
user_reviews_indexed = user_reviews.set_index('asin')
products_indexed = products.set_index('asin')[['price']]

# Need to join so that all user reviews have the price of the associated product
joined = user_reviews_indexed.join(products_indexed, how='left')

# This is a 2x2 dataframe
corr = joined[['price', 'overall']].corr(method='pearson').compute()
ans3 = corr['price'].iloc[1].round(2)
# end = time.time()
# print(end - start)
ans3

np.float64(-0.01)

In [30]:
## Question 4
# Time: 11.4 seconds
# start = time.time()
desc = products['price'].describe().compute()
ans4 = {'mean': desc['mean'], 'std': desc['std'], 'min': desc['min'], 'max': desc['max'],'median': desc['50%']}
# end = time.time()
# print(end - start)
ans4

11.433757543563843


{'mean': np.float64(34.937356116908504),
 'std': np.float64(71.26369249877509),
 'min': np.float64(0.0),
 'max': np.float64(999.99),
 'median': np.float64(19.55)}

In [40]:
products['categories'].head()

0                                          [['Books']]
1                          [['Movies & TV', 'Movies']]
2    [['Clothing, Shoes & Jewelry', 'Girls'], ['Clo...
3    [['Sports & Outdoors', 'Other Sports', 'Dance'...
4     [['Sports & Outdoors', 'Other Sports', 'Dance']]
Name: categories, dtype: string

In [65]:
## Question 5
# Time: 13.67 seconds

# Idea: Process the category column from the products table and get first value from each list
# Groupby this processed column and use .count()
# Sort in non-increasing order
   
# start = time.time()
first_cat = products['categories'].dropna().str.extract(r"\[\['([^']+)")[0]
products = products.assign(first_cat=first_cat)
selected = products[['asin', 'first_cat']].groupby(by='first_cat').count().sort_values(by='asin',ascending=False).compute()
ans5 = selected.to_dict()
# end = time.time()
# print(end-start)
selected.head()

13.67153787612915


,asin
first_cat,
Books,2369910
"Clothing, Shoes & Jewelry",1435868
Sports & Outdoors,529989
Electronics,495476
CDs & Vinyl,491713


In [64]:
selected

,asin
first_cat,
#508510,1
Collectible Coins,1
Gospel,2
Celebrate your Birthday with Nickelodeon,2
Publishers,2
...,...
CDs & Vinyl,491713
Electronics,495476
Sports & Outdoors,529989


In [74]:
## Question 6
# Time: 50ish seconds
from dask import compute
start = time.time()
ans6 = 0 if len(products['asin'].unique())==len(user_reviews['asin'].unique()) else 1
print(f"Answer 6: {ans6}")
end = time.time()
print(end - start)

Answer 6: 1
50.004528284072876


In [ ]:
### read in the 'user_reviews.csv' and 'products.csv' files, perform your calculations and place the answers in variables ans1 - ans7.


# substitute 'None' with the outputs from your calculations. 
# The expected output types can be seen in the assertion statements below
ans1 = None
ans2 = None
ans3 = None
ans4 = None
ans5 = None
ans6 = None
ans7 = None

## <font color='red'> DO NOT MODIFY </font>

In [ ]:
end = time.time()

In [ ]:
print(f"execution time = {end-start}s")

In [ ]:
# DO NOT MODIFY
assert type(ans1) == dict, f"answer to question 1 must be a dictionary like {{'reviewerID':0.2, ..}}, got type = {type(ans1)}"
assert type(ans2) == dict, f"answer to question 2 must be a dictionary like {{'asin':0.2, ..}}, got type = {type(ans2)}"
assert type(ans3) == float, f"answer to question 3 must be a float like 0.8, got type = {type(ans3)}"
assert type(ans4) == dict, f"answer to question 4 must be a dictionary like {{'mean':0.4,'max':0.6,'median':0.6...}}, got type = {type(ans4)}"
assert type(ans5) == dict, f"answer to question 5 must be a dictionary, got type = {type(ans5)}"         
assert ans6 == 0 or ans6==1, f"answer to question 6 must be 0 or 1, got value = {ans6}" 
assert ans7 == 0 or ans7==1, f"answer to question 7 must be 0 or 1, got value = {ans7}" 

ans_dict = {
    "q1": ans1,
    "q2": ans2,
    "q3": ans3,
    "q4": ans4,
    "q5": ans5,
    "q6": ans6,
    "q7": ans7,
    "runtime": end-start
}
with open('my_results_PA1.json', 'w') as outfile: json.dump(ans_dict, outfile)         